# Step 12: fairness check for the allocation experiment
Step 11 showed that releasing only the 5 attributes with the highest public-sample mutual information (top-k) helps LOCAL DP at small budgets. A reviewer will ask: would the CENTRAL model also improve if it used only the same 5 attributes? This notebook answers that: central DP-LR (and non-private LR) restricted to the same top-k attributes, compared with local top-k and with central DP-LR on all attributes.
Needs the files `alloc_*.csv` (Step 11) and `util_*.csv` (final pipeline) in the same Drive folder; if they are missing the corresponding columns are skipped.

In [ ]:
!pip -q install diffprivlib
import numpy as np, pandas as pd, warnings, time, os, glob, shutil
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.datasets import fetch_openml
from diffprivlib.models import LogisticRegression as DPLR, GaussianNB as DPGNB
warnings.filterwarnings("ignore")

try:
    from google.colab import drive, files
    drive.mount('/content/drive'); OUT = "/content/drive/MyDrive/dp_paper/final/"; IN_COLAB = True
except Exception:
    OUT = "./final/"; IN_COLAB = False
os.makedirs(OUT + "figures", exist_ok=True)

QUICK = os.environ.get("DP_QUICK") == "1"          # tiny smoke-test mode (ignore in Colab)
SEED = 42; MAX_ROWS = 5000; PUBLIC_FRAC = 0.10; NB_BINS = 5
TOTALS = [10, 25, 50, 100]                          # matched total epsilon
N_DRAWS = 5                                         # noise draws / model seeds
MIA_REPS = 10; REID_DRAWS = 3; POOLS = [100, 500, 2000]; REID_TOTALS = [10, 25, 50, 100]
if QUICK: TOTALS = [25, 50]; N_DRAWS = 2; MIA_REPS = 3; REID_DRAWS = 1; POOLS = [100, 500]; REID_TOTALS = [25, 50]

# optional extra datasets from CSV files you upload to Colab:  name: (path, target_column, positive_value)
# Lending Club (small, 9,578 loans, 2007-2010): download loan_data.csv from Kaggle, upload to Colab, then use:
#   EXTRA_CSV = {"lending_club": ("/content/loan_data.csv", "not.fully.paid", 1)}
EXTRA_CSV = {}

In [ ]:
# ---------------- dataset loaders ----------------
def load_loan():
    d = pd.read_csv("https://raw.githubusercontent.com/shrikant-temburwar/Loan-Prediction-Dataset/master/train.csv")
    return d.drop(columns=["Loan_ID", "Loan_Status"]), (d.Loan_Status == "Y").astype(int).values
def load_german():
    g = pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/german.csv", header=None)
    return g.iloc[:, :-1], (g.iloc[:, -1] == 2).astype(int).values
def load_adult():
    a = fetch_openml("adult", version=2, as_frame=True, parser="auto").frame
    return a.drop(columns=["class"]), (a["class"] == ">50K").astype(int).values
def load_taiwan():
    t = fetch_openml(data_id=42477, as_frame=True, parser="auto").frame
    return t.iloc[:, :-1], pd.factorize(t.iloc[:, -1])[0].astype(int)
def load_openml(i):
    f = fetch_openml(data_id=i, as_frame=True, parser="auto").frame
    t = f.columns[-1]; lab = f[t].astype(str)
    return f.drop(columns=[t]), (lab != lab.mode()[0]).astype(int).values
def load_lending_club():
    d = pd.read_csv("https://raw.githubusercontent.com/Amitabh1989/random_forest/master/loan_data.csv")   # Lending Club 2007-2010, 9,578 loans
    return d.drop(columns=["not.fully.paid"]), d["not.fully.paid"].astype(int).values
def load_openml_xy(i):
    b = fetch_openml(data_id=i, as_frame=True, parser="auto")        # uses the dataset's own target column
    X = b.data; lab = b.target.astype(str)
    return X, (lab != lab.mode()[0]).astype(int).values              # positive class = minority (default)
def load_csv(path, target, positive=None):
    d = pd.read_csv(path); d = d.loc[:, ~d.columns.astype(str).str.contains("^Unnamed")]
    lab = d[target]
    y = (lab == positive).astype(int).values if positive is not None else (lab.astype(str) != lab.astype(str).mode()[0]).astype(int).values
    return d.drop(columns=[target]), y

ALL = {"loan": load_loan, "german": load_german, "adult": load_adult, "taiwan": load_taiwan,
       "credit_approval": lambda: load_openml(29), "bank_marketing": lambda: load_openml(1461), "diabetes": lambda: load_openml(37),
       "gmc": lambda: load_openml_xy(46468),
       "lending_club": load_lending_club}          # Give Me Some Credit (OpenML 46468, 45,000 rows, ~6.7% default)
MAX_ROWS_OVERRIDE = {"gmc": 15000}
for k, (p, t, pos) in EXTRA_CSV.items(): ALL[k] = (lambda p=p, t=t, pos=pos: load_csv(p, t, pos))
DATASETS = {k: ALL[k] for k in (["loan", "german", "lending_club"] if QUICK else list(ALL))}

def prepare(name):
    X, y = DATASETS[name](); y = np.asarray(y).astype(int)
    cap = MAX_ROWS_OVERRIDE.get(name, MAX_ROWS)
    if len(X) > cap:
        idx = np.random.default_rng(SEED).choice(len(X), cap, replace=False); X, y = X.iloc[idx].reset_index(drop=True), y[idx]
    X = X.copy(); num, cat = [], []
    for c in X.columns:
        if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 10: X[c] = X[c].astype(float); num.append(str(c))
        else: X[c] = pd.factorize(X[c].astype(str))[0]; cat.append(str(c))
    X.columns = X.columns.astype(str)
    perm = np.random.default_rng(SEED).permutation(len(X)); n_pub = max(int(PUBLIC_FRAC * len(X)), 50)
    Xpub = X.iloc[perm[:n_pub]].reset_index(drop=True); Xpriv = X.iloc[perm[n_pub:]].reset_index(drop=True); ypriv = y[perm[n_pub:]]
    for c in num:
        m = Xpub[c].median(); Xpub[c] = Xpub[c].fillna(m); Xpriv[c] = Xpriv[c].fillna(m)
    bounds = {}
    for c in num:
        lo, hi = float(Xpub[c].quantile(.01)), float(Xpub[c].quantile(.99))
        bounds[c] = (lo, hi if hi > lo else lo + 1.0)
    dom = {c: int(X[c].max()) + 1 for c in cat}
    edges = {c: np.quantile(Xpub[c], np.linspace(0, 1, NB_BINS + 1)[1:-1]) for c in num}
    ypub = y[perm[:n_pub]]
    return dict(name=name, X=Xpriv, y=ypriv, Xpub=Xpub, ypub=ypub, num=num, cat=cat, bounds=bounds, dom=dom, edges=edges, d=len(num) + len(cat))

In [ ]:
# ---------------- mechanisms and designs ----------------
def scaled(X, D, c):
    lo, hi = D["bounds"][c]; return (X[c].clip(lo, hi).values - lo) / (hi - lo)       # in [0,1]; sensitivity 1
def onehot(codes, K):
    M = np.zeros((len(codes), K)); M[np.arange(len(codes)), np.asarray(codes).astype(int)] = 1; return M
def rr_codes(col, k, eps, rng):
    col = np.asarray(col).astype(int)
    if k == 1: return col
    p = np.exp(eps) / (np.exp(eps) + k - 1); keep = rng.random(len(col)) < p
    r = rng.integers(0, k - 1, len(col)); return np.where(keep, col, r + (r >= col))
def chan(k, eps):
    p = np.exp(eps) / (np.exp(eps) + k - 1); q = (1 - p) / (k - 1); M = np.full((k, k), q); np.fill_diagonal(M, p); return M

def design(D, eps=None, rng=None):
    # eps=None -> clean design; otherwise Laplace(1/eps) on scaled numerics + randomized response on categoricals
    X = D["X"]; cols, names = [], []
    for c in D["num"]:
        v = scaled(X, D, c)
        if eps is not None: v = v + rng.laplace(0, 1 / eps, len(v))
        cols.append(v[:, None]); names.append(c)
    for c in D["cat"]:
        K = D["dom"][c]; codes = X[c].values.astype(int)
        if eps is not None: codes = rr_codes(codes, K, eps, rng)
        cols.append(onehot(codes, K)); names += [f"{c}_{k}" for k in range(K)]
    return pd.DataFrame(np.hstack(cols), columns=names)

def cv_auc(D, fit_score):
    y = D["y"]; a = []
    for tr, te in StratifiedKFold(10, shuffle=True, random_state=SEED).split(D["X"], y):
        a.append(roc_auc_score(y[te], fit_score(tr, te)))
    return float(np.mean(a))

def central(D, Z, make, reps):
    y = D["y"]; v = [cv_auc(D, lambda tr, te, s=s: make(s).fit(Z.iloc[tr], y[tr]).predict_proba(Z.iloc[te])[:, 1]) for s in range(reps)]
    return np.mean(v), np.std(v)

def local_lr(D, Zc, eps, rng):
    y = D["y"]; Zn = design(D, eps, rng); yn = rr_codes(y, 2, eps, rng)      # label randomized too
    def fs(tr, te):
        m = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=500)).fit(Zn.iloc[tr], yn[tr])
        return m.predict_proba(Zc.iloc[te])[:, 1]                           # clean test records
    return cv_auc(D, fs)

def local_nb(D, eps, rng):
    X = D["X"]; y = D["y"]
    Cc = {c: X[c].values.astype(int) for c in D["cat"]}
    for c in D["num"]: Cc[c] = np.clip(np.searchsorted(D["edges"][c], X[c].clip(*D["bounds"][c]).values), 0, NB_BINS - 1)
    K = {**D["dom"], **{c: NB_BINS for c in D["num"]}}
    Cn = {c: rr_codes(Cc[c], K[c], eps, rng) for c in Cc}; yn = rr_codes(y, 2, eps, rng)
    My = np.linalg.inv(chan(2, eps)); Mx = {c: np.linalg.inv(chan(K[c], eps)) for c in Cc if K[c] > 1}
    def fs(tr, te):
        ny = np.clip(My @ np.bincount(yn[tr], minlength=2), 1e-3, None); score = np.tile(np.log(ny / ny.sum()), (len(te), 1))
        for c in Cc:
            if K[c] == 1: continue
            N = np.zeros((2, K[c])); np.add.at(N, (yn[tr], Cn[c][tr]), 1)
            T = np.clip(My @ N @ Mx[c], 0, None) + 1.0; P = T / T.sum(1, keepdims=True)    # debiased joint counts + smoothing
            score += np.log(P[:, Cc[c][te]]).T
        return score[:, 1] - score[:, 0]
    return cv_auc(D, fs)

In [ ]:
from sklearn.metrics import mutual_info_score
def attr_codes(Xd, D, c):
    if c in D["num"]: return np.clip(np.searchsorted(D["edges"][c], Xd[c].clip(*D["bounds"][c]).values), 0, NB_BINS - 1)
    return Xd[c].values.astype(int)
def public_mi(D): return {c: mutual_info_score(D["ypub"], attr_codes(D["Xpub"], D, c)) for c in D["num"] + D["cat"]}
def design_sub(D, attrs):
    X = D["X"]; cols, names = [], []
    for c in D["num"]:
        if c in attrs: cols.append(scaled(X, D, c)[:, None]); names.append(c)
    for c in D["cat"]:
        if c in attrs: K = D["dom"][c]; cols.append(onehot(X[c].values.astype(int), K)); names += [f"{c}_{k}" for k in range(K)]
    return pd.DataFrame(np.hstack(cols), columns=names)

def run_ctk(D, k=5):
    mi = public_mi(D); attrs = D["num"] + D["cat"]; sel = sorted(attrs, key=lambda c: -mi[c])[:min(k, len(attrs))]
    Z = design_sub(D, sel); B = np.sqrt(len(sel)); rows = []
    m, s = central(D, Z, lambda sd: LogisticRegression(C=1.0, max_iter=500), 1)
    rows.append(dict(dataset=D["name"], method="LR_nonprivate_topk", total_eps=np.nan, auc=m, sd=0.0, selected=";".join(sel)))
    for t in TOTALS:
        m, s = central(D, Z, lambda sd: DPLR(epsilon=t, data_norm=B, C=1.0, random_state=sd), N_DRAWS)
        rows.append(dict(dataset=D["name"], method="central_DPLR_topk", total_eps=t, auc=m, sd=s, selected=";".join(sel)))
    return rows

for name in DATASETS:
    path = OUT + f"ctk_{name}.csv"
    if os.path.exists(path): print(name, "already done"); continue
    try:
        t0 = time.time(); D = prepare(name); pd.DataFrame(run_ctk(D)).to_csv(path, index=False); print(name, "done in", round(time.time() - t0), "s")
    except Exception as ex:
        import traceback; traceback.print_exc(); print(name, "FAILED:", repr(ex))

In [ ]:
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
C = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(OUT + "ctk_*.csv"))], ignore_index=True)
U = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(OUT + "util_*.csv"))], ignore_index=True) if glob.glob(OUT + "util_*.csv") else pd.DataFrame()
A = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(OUT + "alloc_*.csv"))], ignore_index=True) if glob.glob(OUT + "alloc_*.csv") else pd.DataFrame()
print("Selected attributes per dataset:"); print(C.drop_duplicates("dataset")[["dataset", "selected"]].to_string(index=False))
rows = []
for t in TOTALS:
    for ds in C.dataset.unique():
        r = dict(total_eps=t, dataset=ds)
        r["central_topk"] = C[(C.dataset == ds) & (C.method == "central_DPLR_topk") & (C.total_eps == t)].auc.mean()
        if len(U): r["central_all"] = U[(U.dataset == ds) & (U.method == "central_DPLR") & (U.total_eps == t)].auc.mean()
        if len(A):
            for sch, mdl in [("uniform", "LR"), ("topk", "LR"), ("topk", "NB")]:
                r[f"local_{sch}_{mdl}"] = A[(A.dataset == ds) & (A.scheme == sch) & (A.model == mdl) & (A.total_eps == t)].auc.mean()
        rows.append(r)
T = pd.DataFrame(rows); T.to_csv(OUT + "ctk_summary.csv", index=False)
for t in TOTALS:
    g = T[T.total_eps == t].set_index("dataset").drop(columns="total_eps"); print(f"\n=== AUC at total epsilon = {t} ==="); print(g.round(3).to_string())
    if "central_all" in g: print(f"central top-k minus central all: mean {np.nanmean(g.central_topk - g.central_all):+.3f}, improved on {(g.central_topk > g.central_all).sum()}/{len(g)}")
    for col in [c for c in g.columns if c.startswith("local_topk")]:
        d_ = g[col] - g.central_topk
        print(f"{col} minus central top-k: mean {d_.mean():+.3f}, local higher on {(d_ > 0).sum()}/{len(d_)}")